# Side-by-side pose comparison (YOLO vs MediaPipe vs OpenPose)

This notebook renders a 3-panel video for slides.

**Inputs**: three `.npz` pose files that correspond to the same original video.

We match them using `meta_json['video_relpath']` (preferred) or by manually specifying the relpath.


In [35]:
import sys
from pathlib import Path

# Notebook is: CV/code/data_population/presentation.ipynb
# We want:     CV/code on sys.path
CODE_ROOT = Path.cwd().resolve().parents[0].parent  # -> CV/code

# If your working directory is already data_population, this works.
# More robust:
if (Path.cwd().resolve() / "data_population").exists():
    # you are in CV/code
    CODE_ROOT = Path.cwd().resolve()
else:
    # you are likely in CV/code/data_population
    CODE_ROOT = Path.cwd().resolve().parent

if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

print("Added to PYTHONPATH:", CODE_ROOT)
print("sys.path[0]:", sys.path[0])


from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import cv2


Added to PYTHONPATH: D:\Brad\School\UofT\Year4\CSC494_eng\aps490-capstone-kite\CV\code
sys.path[0]: D:\Brad\School\UofT\Year4\CSC494_eng\aps490-capstone-kite\CV\code


## CONFIG

Set your three output roots (one per backend), and which pipeline stage tag you want.

Your main.py currently produces files like:
`<out_root>/<relpath_without_ext>_<tag>.npz`

Example tag: `raw_interp_smooth`


In [36]:
# ----------------------------
# EDIT THESE
# ----------------------------
VIDEO_PATH = r"C:\Users\brad\OneDrive - UHN\Li, Yue (Sophia)'s files - WinterLab videos\raw videos to rename the gopro files\videos_renamed\2025-02-19\sub349\idapt797_sub349_DP_13_12-13-01.MP4"  # <-- absolute path to the raw mp4

# Processing toggles (these define BOTH the pipeline AND the label tag)
DO_INTERP = False
DO_SMOOTH = False

# Render options
if DO_INTERP:
    FPS = 120
else:
    FPS = 30
CONF_THR = 0.05
PERSON_INDEX = 0

# Panel/canvas
PANEL_W = 640
PANEL_H = 720
CANVAS_W = PANEL_W * 3
CANVAS_H = PANEL_H

DRAW_ON_BLACK = True
LABEL_KPT_INDICES = False

# Save everything here
SAVE_DIR = Path(r"D:\temp")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

OUT_BASENAME = "pose_compare_3panel"
TMP_MP4 = SAVE_DIR / f"{OUT_BASENAME}_tmp.mp4"
FINAL_MP4 = SAVE_DIR / f"{OUT_BASENAME}.mp4"


## Skeleton edges (auto-picked based on K)

YOLO will be COCO17 (K=17)
OpenPose will be BODY25 (K=25)
MediaPipe will be MP33 (K=33)


In [37]:
COCO17_EDGES: List[tuple[int, int]] = [
    (5, 6),
    (5, 7), (7, 9),
    (6, 8), (8, 10),
    (11, 12),
    (5, 11), (6, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16),
    (0, 1), (0, 2), (1, 3), (2, 4),
    (0, 5), (0, 6),
]

BODY25_EDGES: List[tuple[int, int]] = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (1, 5), (5, 6), (6, 7),
    (1, 8), (8, 9), (9, 10), (10, 11),
    (8, 12), (12, 13), (13, 14),
    (0, 15), (15, 17),
    (0, 16), (16, 18),
    (14, 19), (19, 20), (20, 21),
    (11, 22), (22, 23), (23, 24),
]

MEDIAPIPE33_EDGES: List[tuple[int, int]] = [
    (11, 12), (11, 23), (12, 24), (23, 24),
    (11, 13), (13, 15), (15, 17), (15, 19), (15, 21), (17, 19), (19, 21),
    (12, 14), (14, 16), (16, 18), (16, 20), (16, 22), (18, 20), (20, 22),
    (23, 25), (25, 27), (27, 29), (29, 31), (27, 31),
    (24, 26), (26, 28), (28, 30), (30, 32), (28, 32),
    (0, 1), (0, 2), (1, 3), (2, 4), (0, 11), (0, 12),
]

def pick_edges(K: int) -> List[tuple[int, int]]:
    if K == 17:
        return COCO17_EDGES
    if K == 25:
        return BODY25_EDGES
    if K == 33:
        return MEDIAPIPE33_EDGES
    return []

def guess_format(K: int) -> str:
    return {17: "coco17/yolo", 25: "openpose_body25", 33: "mediapipe33"}.get(K, f"unknown(K={K})")


## Helpers: load NPZ, normalize, draw panel


In [38]:
def load_npz(npz_path: Path) -> Tuple[np.ndarray, dict]:
    data = np.load(npz_path, allow_pickle=True)
    if "poses" not in data:
        raise ValueError(f"{npz_path} missing 'poses' array. Keys: {list(data.keys())}")
    poses = data["poses"]

    meta = {}
    if "meta_json" in data:
        try:
            meta = json.loads(data["meta_json"].item())
        except Exception:
            meta = {}
    return poses, meta

def to_TNKC(poses: np.ndarray) -> np.ndarray:
    arr = np.asarray(poses)
    if arr.ndim == 3:
        return arr[:, None, :, :]  # (T,1,K,C)
    if arr.ndim == 4:
        return arr
    raise ValueError(f"Unsupported pose shape: {arr.shape}")

def normalize_if_needed(xy: np.ndarray, W: int, H: int) -> np.ndarray:
    mx = float(np.nanmax(xy[..., 0]))
    my = float(np.nanmax(xy[..., 1]))
    if mx <= 2.0 and my <= 2.0:
        xy = xy.copy()
        xy[..., 0] *= W
        xy[..., 1] *= H
    return xy

def draw_panel_frame(
    t: int,
    poses_tnkc: np.ndarray,
    base_frame_bgr: np.ndarray,   # frame to draw on (already resized to panel size)
    conf_thr: float,
    person_index: int,
    title: str,
    edges: List[tuple[int, int]],
    label_indices: bool,
    *,
    src_w: Optional[int] = None,  # NEW: source width of pose coords
    src_h: Optional[int] = None,  # NEW: source height of pose coords
) -> np.ndarray:
    img = base_frame_bgr.copy()
    H, W = img.shape[:2]

    T, N, K, C = poses_tnkc.shape
    if t < 0 or t >= T:
        return img
    if person_index < 0 or person_index >= N:
        return img

    pts = poses_tnkc[t, person_index]
    xy = pts[:, :2].astype(np.float32)
    conf = pts[:, 2].astype(np.float32) if C >= 3 else None

    # 1) If coords look normalized, scale to panel directly
    mx = float(np.nanmax(xy[:, 0])) if xy.size else 0.0
    my = float(np.nanmax(xy[:, 1])) if xy.size else 0.0
    if mx <= 2.0 and my <= 2.0:
        xy = xy.copy()
        xy[:, 0] *= W
        xy[:, 1] *= H
    else:
        # 2) Otherwise assume pixel coords in src resolution, scale to panel
        if src_w is not None and src_h is not None and src_w > 0 and src_h > 0:
            sx = W / float(src_w)
            sy = H / float(src_h)
            xy = xy.copy()
            xy[:, 0] *= sx
            xy[:, 1] *= sy
        # else: fallback = assume already in panel pixel space

    # bones
    for a, b in edges:
        if a >= K or b >= K:
            continue
        if conf is not None and (conf[a] < conf_thr or conf[b] < conf_thr):
            continue
        ax, ay = xy[a]
        bx, by = xy[b]
        if not np.isfinite([ax, ay, bx, by]).all():
            continue
        cv2.line(img, (int(ax), int(ay)), (int(bx), int(by)), (0, 255, 0), 2, cv2.LINE_AA)

    # joints
    for k in range(K):
        if conf is not None and conf[k] < conf_thr:
            continue
        x, y = xy[k]
        if not np.isfinite([x, y]).all():
            continue
        cv2.circle(img, (int(x), int(y)), 3, (0, 0, 255), -1, cv2.LINE_AA)
        if label_indices:
            cv2.putText(img, str(k), (int(x) + 4, int(y) - 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)

    cv2.rectangle(img, (0, 0), (W, 48), (0, 0, 0), -1)
    cv2.putText(img, title, (14, 34), cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                (255, 255, 255), 2, cv2.LINE_AA)

    return img



def reencode_h264(src: Path, dst: Path) -> None:
    subprocess.run(
        [
            "ffmpeg", "-y",
            "-i", str(src),
            "-c:v", "libx264",
            "-profile:v", "baseline",
            "-level", "3.1",
            "-pix_fmt", "yuv420p",
            "-movflags", "+faststart",
            str(dst),
        ],
        check=True,
    )


In [39]:
from dataclasses import asdict

# ---- import your pipeline modules (same as main.py) ----
from pose import PoseExtractor, PoseBackendConfig
from preprocessing.pose_interpolation import InterpolationConfig, interpolate_pose_sequence
from preprocessing.pose_smoothing import SmoothingConfig, EMAPoseSmoother


def stage_tag(do_interp: bool, do_smooth: bool) -> str:
    if do_interp and do_smooth:
        return "raw_interp_smooth"
    if do_interp and not do_smooth:
        return "raw_interp"
    if (not do_interp) and do_smooth:
        return "raw_smooth"
    return "raw"


# ---- These should match the config in your main.py ----
# OpenPose
OPENPOSE_EXE = r"D:\Brad\School\UofT\Year4\CSC494_eng\aps490-capstone-kite\CV\frameworks\openpose\bin\OpenPoseDemo.exe"
OPENPOSE_MODEL_FOLDER = r"D:\Brad\School\UofT\Year4\CSC494_eng\aps490-capstone-kite\CV\frameworks\openpose\models"
OPENPOSE_MODEL_POSE = "BODY_25"
OPENPOSE_NUMBER_PEOPLE_MAX = 1

# YOLO
YOLO_MODEL_PATH = r"D:\Brad\School\UofT\Year4\CSC494_eng\aps490-capstone-kite\CV\models\yolo26x-pose.pt"
YOLO_DEVICE = "cuda:0"           # change if you want
YOLO_BATCH_SIZE = 8
YOLO_VERBOSE = False

# MediaPipe
MP_MODEL_PATH = r"D:\Brad\School\UofT\Year4\CSC494_eng\aps490-capstone-kite\CV\models\pose_landmarker_heavy.task"
MP_MIN_DET = 0.5
MP_MIN_PRES = 0.5
MP_MIN_TRACK = 0.5


def build_pose_extractor(backend: str) -> PoseExtractor:
    name = backend.lower()

    if name == "openpose":
        cfg = PoseBackendConfig(
            name="openpose",
            op_exe_path=OPENPOSE_EXE,
            op_model_folder=OPENPOSE_MODEL_FOLDER,
            op_model_pose=OPENPOSE_MODEL_POSE,
            op_number_people_max=OPENPOSE_NUMBER_PEOPLE_MAX,
        )
        return PoseExtractor(cfg)

    if name == "yolo":
        cfg = PoseBackendConfig(
            name="yolo",
            yolo_model_path=YOLO_MODEL_PATH,
            yolo_device=YOLO_DEVICE,
            yolo_batch_size=YOLO_BATCH_SIZE,
            yolo_verbose=YOLO_VERBOSE,
            yolo_num_kpts=17,
        )
        return PoseExtractor(cfg)

    if name == "mediapipe":
        cfg = PoseBackendConfig(
            name="mediapipe",
            mp_model_path=MP_MODEL_PATH,
            mp_num_poses=1,
            mp_min_det_conf=MP_MIN_DET,
            mp_min_presence_conf=MP_MIN_PRES,
            mp_min_track_conf=MP_MIN_TRACK,
        )
        return PoseExtractor(cfg)

    raise ValueError(f"Unknown backend={backend!r}")


def run_pipeline_and_save_npz(
    *,
    backend: str,
    video_abs: str,
    do_interp: bool,
    do_smooth: bool,
    out_dir: Path,
    conf_thr: float,
) -> Path:
    extractor = build_pose_extractor(backend)

    # match your main.py defaults
    interp_cfg = InterpolationConfig(
        scale_factor=4,      # FPS_SCALE from main.py
        mode="linear",       # INTERP_MODE from main.py
        conf_thr=conf_thr,
        frame_min_kpts=10,
        frame_min_frac=0.0,
        clip_to_frame=None,
    )

    smoother = EMAPoseSmoother(SmoothingConfig(
        alpha=0.7,          # EMA_ALPHA from main.py
        conf_thr=conf_thr,
        smooth_conf=False,
        missing_policy="hold",
        clip_to_frame=None,
    ))

    tag = stage_tag(do_interp, do_smooth)

    # output name in D:\temp
    stem = Path(video_abs).stem
    out_path = out_dir / f"{stem}_{backend}_{tag}.npz"

    # generate (skip if already exists)
    if out_path.exists():
        print(f"[gen] exists: {out_path.name}")
        return out_path

    print(f"[gen] extracting backend={backend} -> {out_path.name}")
    poses_raw, meta = extractor.extract_pose_from_video(video_abs, conf_thr=conf_thr)

    poses_stage = poses_raw
    poses_interp = None
    poses_smooth = None

    if do_interp:
        poses_interp = interpolate_pose_sequence(poses_stage, interp_cfg)
        poses_stage = poses_interp

    if do_smooth:
        poses_smooth = smoother.smooth_sequence(poses_stage)
        poses_stage = poses_smooth

    payload_meta = {
        **meta,
        "backend": backend,
        "video_path": video_abs,
        "pipeline": {"do_interp": bool(do_interp), "do_smooth": bool(do_smooth)},
        "interp_config": asdict(interp_cfg) if do_interp else None,
        "smooth_config": asdict(smoother.config) if do_smooth else None,
        "poses_raw_shape": list(np.asarray(poses_raw).shape),
        "poses_interp_shape": list(np.asarray(poses_interp).shape) if poses_interp is not None else None,
        "poses_smooth_shape": list(np.asarray(poses_smooth).shape) if poses_smooth is not None else None,
        "poses_saved_stage": tag,
    }

    np.savez_compressed(
        out_path,
        poses=np.asarray(poses_stage, dtype=np.float32),
        meta_json=json.dumps(payload_meta),
    )

    return out_path


## Locate the three matching NPZ files

We construct the expected output path based on the extraction logic:
`<out_root>/<rel_no_ext>_<tag>.npz`


In [40]:
video_abs = os.path.abspath(os.path.expanduser(VIDEO_PATH))
if not os.path.isfile(video_abs):
    raise FileNotFoundError(video_abs)

tag = stage_tag(DO_INTERP, DO_SMOOTH)
print("Pipeline tag:", tag)

npz_yolo = run_pipeline_and_save_npz(
    backend="yolo",
    video_abs=video_abs,
    do_interp=DO_INTERP,
    do_smooth=DO_SMOOTH,
    out_dir=SAVE_DIR,
    conf_thr=CONF_THR,
)

npz_mp = run_pipeline_and_save_npz(
    backend="mediapipe",
    video_abs=video_abs,
    do_interp=DO_INTERP,
    do_smooth=DO_SMOOTH,
    out_dir=SAVE_DIR,
    conf_thr=CONF_THR,
)

npz_op = run_pipeline_and_save_npz(
    backend="openpose",
    video_abs=video_abs,
    do_interp=DO_INTERP,
    do_smooth=DO_SMOOTH,
    out_dir=SAVE_DIR,
    conf_thr=CONF_THR,
)

print("YOLO:", npz_yolo)
print("MP  :", npz_mp)
print("OP  :", npz_op)


Pipeline tag: raw
[gen] exists: idapt797_sub349_DP_13_12-13-01_yolo_raw.npz
[gen] exists: idapt797_sub349_DP_13_12-13-01_mediapipe_raw.npz
[gen] exists: idapt797_sub349_DP_13_12-13-01_openpose_raw.npz
YOLO: D:\temp\idapt797_sub349_DP_13_12-13-01_yolo_raw.npz
MP  : D:\temp\idapt797_sub349_DP_13_12-13-01_mediapipe_raw.npz
OP  : D:\temp\idapt797_sub349_DP_13_12-13-01_openpose_raw.npz


## Load poses + print basic metadata


In [41]:
poses_yolo_raw, meta_yolo = load_npz(npz_yolo)
poses_mp_raw, meta_mp = load_npz(npz_mp)
poses_op_raw, meta_op = load_npz(npz_op)

poses_yolo = to_TNKC(poses_yolo_raw)
poses_mp = to_TNKC(poses_mp_raw)
poses_op = to_TNKC(poses_op_raw)

def info(name: str, poses: np.ndarray, meta: dict):
    T, N, K, C = poses.shape
    print(f"[{name}] T={T} N={N} K={K} C={C} format={guess_format(K)}")
    vr = meta.get("video_relpath")
    backend = meta.get("backend")
    stage = meta.get("poses_saved_stage")
    if vr or backend or stage:
        print(f"   meta: backend={backend} stage={stage} video_relpath={vr}")

info("YOLO", poses_yolo, meta_yolo)
info("MEDIAPIPE", poses_mp, meta_mp)
info("OPENPOSE", poses_op, meta_op)


[YOLO] T=485 N=1 K=17 C=3 format=coco17/yolo
   meta: backend=yolo stage=raw video_relpath=None
[MEDIAPIPE] T=485 N=1 K=33 C=3 format=mediapipe33
   meta: backend=mediapipe stage=raw video_relpath=None
[OPENPOSE] T=485 N=1 K=25 C=3 format=openpose_body25
   meta: backend=openpose stage=raw video_relpath=None


## Align time (T)

Different backends can output slightly different frame counts.

For slides, simplest is: **use `min_T` across all three**.

If you prefer padding/holding to `max_T`, we can add that after.


In [42]:
T_min = min(poses_yolo.shape[0], poses_mp.shape[0], poses_op.shape[0])
print("Using T_min =", T_min)

poses_yolo = poses_yolo[:T_min]
poses_mp = poses_mp[:T_min]
poses_op = poses_op[:T_min]


Using T_min = 485


## Render 3-panel MP4


In [43]:
SAVE_DIR.mkdir(parents=True, exist_ok=True)

edges_yolo = pick_edges(poses_yolo.shape[2])
edges_mp = pick_edges(poses_mp.shape[2])
edges_op = pick_edges(poses_op.shape[2])

# build a human-readable pipeline label
if DO_INTERP and DO_SMOOTH:
    pipeline_label = "interp + smooth"
elif DO_INTERP and not DO_SMOOTH:
    pipeline_label = "interp"
elif (not DO_INTERP) and DO_SMOOTH:
    pipeline_label = "smooth"
else:
    pipeline_label = "raw"

# ---- open source video (for the YOLO panel background) ----
cap = cv2.VideoCapture(str(video_abs))
if not cap.isOpened():
    raise RuntimeError(f"Failed to open video: {video_abs}")

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
start_frame = int(total_frames * 0.5) if total_frames > 0 else 0
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

# ---- choose T range safely (poses + remaining video frames) ----
T_yolo = poses_yolo.shape[0]
T_mp   = poses_mp.shape[0]
T_op   = poses_op.shape[0]
T_poses_min = min(T_yolo, T_mp, T_op)

remaining_video = max(0, total_frames - start_frame) if total_frames > 0 else T_poses_min
T_min = min(T_poses_min, remaining_video)

if T_min <= 0:
    cap.release()
    raise RuntimeError("T_min computed as 0. Check pose lengths and video frame count.")

# black background for MP/OP panels
black_panel = np.zeros((PANEL_H, PANEL_W, 3), dtype=np.uint8)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(TMP_MP4), fourcc, FPS, (CANVAS_W, CANVAS_H), isColor=True)
if not writer.isOpened():
    cap.release()
    raise RuntimeError("VideoWriter failed to open. Try output .avi or ensure codecs installed.")


src_w_yolo = meta_yolo.get("width")
src_h_yolo = meta_yolo.get("height")

src_w_mp = meta_mp.get("width")
src_h_mp = meta_mp.get("height")

src_w_op = meta_op.get("width")
src_h_op = meta_op.get("height")

print("src sizes:",
    "yolo", (src_w_yolo, src_h_yolo),
    "mp", (src_w_mp, src_h_mp),
    "op", (src_w_op, src_h_op))



for t in range(T_min):
    ok, frame_bgr = cap.read()
    if not ok:
        break

    # panel 1 background = actual video frame
    base_yolo = cv2.resize(frame_bgr, (PANEL_W, PANEL_H), interpolation=cv2.INTER_AREA)


    f1 = draw_panel_frame(
        t=t,
        poses_tnkc=poses_yolo,
        base_frame_bgr=base_yolo,
        conf_thr=CONF_THR,
        person_index=PERSON_INDEX,
        title=f"YOLO (COCO17) | {pipeline_label}",
        edges=edges_yolo,
        label_indices=LABEL_KPT_INDICES,
        src_w=src_w_yolo,
        src_h=src_h_yolo,
        )

    f2 = draw_panel_frame(
        t=t,
        poses_tnkc=poses_mp,
        base_frame_bgr=black_panel,
        conf_thr=CONF_THR,
        person_index=PERSON_INDEX,
        title=f"MediaPipe (33) | {pipeline_label}",
        edges=edges_mp,
        label_indices=LABEL_KPT_INDICES,
        src_w=src_w_mp,
        src_h=src_h_mp,
    )

    f3 = draw_panel_frame(
        t=t,
        poses_tnkc=poses_op,
        base_frame_bgr=black_panel,
        conf_thr=CONF_THR,
        person_index=PERSON_INDEX,
        title=f"OpenPose (BODY25) | {pipeline_label}",
        edges=edges_op,
        label_indices=LABEL_KPT_INDICES,
        src_w=src_w_op,
        src_h=src_h_op,
    )

    canvas = np.concatenate([f1, f2, f3], axis=1)

    # frame counter bottom-left
    cv2.putText(
        canvas,
        f"{t+1}/{T_min}",
        (20, CANVAS_H - 20),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2,
        cv2.LINE_AA,
    )

    writer.write(canvas)

writer.release()
cap.release()

print("Wrote tmp:", TMP_MP4)
print("Re-encoding ->", FINAL_MP4)
reencode_h264(TMP_MP4, FINAL_MP4)

try:
    TMP_MP4.unlink()
except Exception:
    pass

print("DONE:", FINAL_MP4)


src sizes: yolo (1920, 1080) mp (1920, 1080) op (None, None)
Wrote tmp: D:\temp\pose_compare_3panel_tmp.mp4
Re-encoding -> D:\temp\pose_compare_3panel.mp4
DONE: D:\temp\pose_compare_3panel.mp4
